# Session 12 Lab: AI App Development
**Course:** Noob to AI Expert | **Track:** Intermediate

Build a FastAPI backend and Streamlit frontend, then prepare for Streamlit Cloud deployment.

In [ ]:
!pip install fastapi uvicorn anthropic streamlit requests python-dotenv -q
print('✅ Ready')

## Part 1: FastAPI Backend (main.py)
Save this as `main.py` and run: `uvicorn main:app --reload`

In [ ]:
# Save as main.py
backend_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import anthropic, os

app = FastAPI(title="AI FAQ Bot")
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

SYSTEM = """You are a helpful FAQ assistant about Python and AI.
Answer concisely. If uncertain, say so."""

class ChatRequest(BaseModel):
    message: str
    history: list[dict] = []

class ChatResponse(BaseModel):
    reply: str
    tokens_used: int

@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    if not req.message.strip():
        raise HTTPException(400, "Empty message")
    messages = req.history + [{"role": "user", "content": req.message}]
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001", max_tokens=512,
        system=SYSTEM, messages=messages
    )
    return ChatResponse(reply=resp.content[0].text,
                        tokens_used=resp.usage.input_tokens + resp.usage.output_tokens)

@app.get("/health")
async def health(): return {"status": "ok"}
'''

with open('main.py', 'w') as f:
    f.write(backend_code)
print('✅ main.py created')

## Part 2: Streamlit Frontend (app.py)

In [ ]:
frontend_code = '''
import streamlit as st, requests

st.set_page_config(page_title="AI FAQ Bot", page_icon="🤖")
st.title("🤖 AI FAQ Bot")
st.caption("Ask me anything about Python and AI!")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if prompt := st.chat_input("Ask a question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                r = requests.post("http://localhost:8000/chat",
                                  json={"message": prompt, "history": st.session_state.messages[:-1]},
                                  timeout=30)
                reply = r.json()["reply"]
                tokens = r.json()["tokens_used"]
            except Exception as e:
                reply, tokens = f"Error: {e}", 0
        st.write(reply)
        st.caption(f"Tokens: {tokens}")
    st.session_state.messages.append({"role": "assistant", "content": reply})
'''

with open('app.py', 'w') as f:
    f.write(frontend_code)
print('✅ app.py created')
print('Run backend: uvicorn main:app --reload')
print('Run frontend: streamlit run app.py')